# End-to-End Sales Forecasting & Demand Intelligence System

## Exploratory Data Analysis, Time Series Modeling, and Demand Segmentation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os
import joblib

warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Create required directories if they don't exist
for d in ['../charts', '../models', '../outputs']:
    os.makedirs(d, exist_ok=True)

## TASK 1: DATA LOADING & EDA

### Data Loading and Parsing

In [ ]:
# Load dataset
df = pd.read_csv('../datasets/train.csv')

# Convert date columns to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='mixed', dayfirst=True)

print("Dataset shape:", df.shape)
df.head()

### Data Quality Checks

In [ ]:
# Check missing values
missing_values = df.isnull().sum()
print("Missing values:\n", missing_values[missing_values > 0])

# Check duplicates
duplicates = df.duplicated().sum()
print("\nDuplicates:", duplicates)

# Data types
print("\nData Types:\n", df.dtypes)

# Outliers check using descriptive statistics
print("\nDescriptive Stats:\n", df.describe())

### Feature Engineering

In [ ]:
# Create new date-related columns
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Quarter'] = df['Order Date'].dt.quarter
df['Week Number'] = df['Order Date'].dt.isocalendar().week
df['Day Name'] = df['Order Date'].dt.day_name()
df['Day of Week'] = df['Order Date'].dt.dayofweek

# Create Season column
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Fall'
    
df['Season'] = df['Month'].apply(get_season)

# Shipping Time
df['Shipping Time'] = (df['Ship Date'] - df['Order Date']).dt.days

# Handle any potential missing values
df.dropna(subset=['Order Date', 'Sales'], inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

### Aggregations

In [ ]:
daily_sales = df.groupby(df['Order Date'].dt.date)['Sales'].sum().reset_index()
daily_sales['Order Date'] = pd.to_datetime(daily_sales['Order Date'])

weekly_sales = df.set_index('Order Date').resample('W')['Sales'].sum().reset_index()
monthly_sales = df.set_index('Order Date').resample('ME')['Sales'].sum().reset_index()

### Exploratory Data Analysis & Visualizations

In [ ]:
def save_chart(fig, name):
    fig.savefig(f'../charts/{name}.png', bbox_inches='tight')

sns.set_theme(style="whitegrid")

# 1. Highest revenue category
plt.figure(figsize=(10, 6))
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
sns.barplot(x=cat_sales.index, y=cat_sales.values, palette='viridis')
plt.title('Total Revenue by Category')
plt.ylabel('Total Sales ($)')
plt.xlabel('Category')
save_chart(plt.gcf(), 'revenue_by_category')
plt.show()

# 2. Most consistent region (lowest coefficient of variation)
plt.figure(figsize=(10, 6))
region_stats = df.groupby('Region')['Sales'].agg(['mean', 'std'])
region_stats['CV'] = region_stats['std'] / region_stats['mean']
sns.barplot(x=region_stats.index, y=region_stats['CV'], palette='coolwarm')
plt.title('Sales Consistency (Coefficient of Variation) by Region')
plt.ylabel('Coefficient of Variation')
plt.xlabel('Region')
save_chart(plt.gcf(), 'consistency_by_region')
plt.show()

# 3. Shipping time by region
plt.figure(figsize=(10, 6))
sns.boxplot(x='Region', y='Shipping Time', data=df, palette='Set2')
plt.title('Shipping Time Distribution by Region')
plt.ylabel('Days')
plt.xlabel('Region')
save_chart(plt.gcf(), 'shipping_time_by_region')
plt.show()

# 4. Seasonality
plt.figure(figsize=(10, 6))
season_sales = df.groupby('Season')['Sales'].mean().reindex(['Spring', 'Summer', 'Fall', 'Winter'])
sns.barplot(x=season_sales.index, y=season_sales.values, palette='Pastel1')
plt.title('Average Sales by Season')
plt.ylabel('Average Sales ($)')
plt.xlabel('Season')
save_chart(plt.gcf(), 'seasonality_sales')
plt.show()

# 5. Top products
plt.figure(figsize=(12, 6))
top_products = df.groupby('Product Name')['Sales'].sum().nlargest(10)
sns.barplot(y=top_products.index, x=top_products.values, palette='Blues_r')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Total Sales ($)')
plt.ylabel('Product Name')
save_chart(plt.gcf(), 'top_products')
plt.show()

# 6. Top customers
plt.figure(figsize=(12, 6))
top_customers = df.groupby('Customer Name')['Sales'].sum().nlargest(10)
sns.barplot(y=top_customers.index, x=top_customers.values, palette='Greens_r')
plt.title('Top 10 Customers by Revenue')
plt.xlabel('Total Sales ($)')
plt.ylabel('Customer Name')
save_chart(plt.gcf(), 'top_customers')
plt.show()

# 7. Top states
plt.figure(figsize=(12, 6))
top_states = df.groupby('State')['Sales'].sum().nlargest(10)
sns.barplot(y=top_states.index, x=top_states.values, palette='Reds_r')
plt.title('Top 10 States by Revenue')
plt.xlabel('Total Sales ($)')
plt.ylabel('State')
save_chart(plt.gcf(), 'top_states')
plt.show()

# 8. Top cities
plt.figure(figsize=(12, 6))
top_cities = df.groupby('City')['Sales'].sum().nlargest(10)
sns.barplot(y=top_cities.index, x=top_cities.values, palette='Purples_r')
plt.title('Top 10 Cities by Revenue')
plt.xlabel('Total Sales ($)')
plt.ylabel('City')
save_chart(plt.gcf(), 'top_cities')
plt.show()

# 9. Top sub-categories
plt.figure(figsize=(12, 6))
top_subcat = df.groupby('Sub-Category')['Sales'].sum().nlargest(10)
sns.barplot(y=top_subcat.index, x=top_subcat.values, palette='Oranges_r')
plt.title('Top 10 Sub-Categories by Revenue')
plt.xlabel('Total Sales ($)')
plt.ylabel('Sub-Category')
save_chart(plt.gcf(), 'top_subcategories')
plt.show()

# 10. Sales Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['Sales'][df['Sales'] < 1000], bins=50, color='blue')
plt.title('Sales Distribution (< $1000)')
plt.xlabel('Sales ($)')
save_chart(plt.gcf(), 'sales_distribution')
plt.show()

### Business Interpretations (EDA)
- **Revenue:** Technology leads in revenue, followed closely by Furniture and Office Supplies.
- **Consistency:** The Central region exhibits higher variance, making it less consistent than the East or West regions.
- **Seasonality:** Fall and Winter seasons typically drive higher sales, indicative of end-of-year holiday buying patterns.
- **Distributions:** Most transactions are of low monetary value, indicating a high volume of smaller orders.

## TASK 2: TIME SERIES ANALYSIS

### Creating Monthly Series & Visualization

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

# Create monthly series
ts_monthly = df.set_index('Order Date').resample('MS')['Sales'].sum()

plt.figure(figsize=(12, 6))
plt.plot(ts_monthly.index, ts_monthly.values, label='Monthly Sales', color='blue')
plt.plot(ts_monthly.index, ts_monthly.rolling(window=12).mean(), label='12-Month Rolling Mean', color='red')
plt.plot(ts_monthly.index, ts_monthly.rolling(window=12).std(), label='12-Month Rolling Std', color='green')
plt.title('Monthly Sales Trend, Rolling Mean & Standard Deviation')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
save_chart(plt.gcf(), 'monthly_sales_trend')
plt.show()

### Time Series Decomposition

In [ ]:
decomposition = seasonal_decompose(ts_monthly.dropna(), model='additive')
fig = decomposition.plot()
fig.set_size_inches(12, 8)
save_chart(fig, 'ts_decomposition')
plt.show()

### Stationarity Check (ADF Test)

In [ ]:
def run_adf_test(series):
    result = adfuller(series.dropna())
    print(f'ADF Statistic: {result[0]}')
    print(f'p-value: {result[1]}')
    for key, value in result[4].items():
        print(f'Critical Value ({key}): {value}')
    return result[1]

print("ADF Test on Original Series:")
p_val = run_adf_test(ts_monthly)

### Stationarity Explained
Stationarity means that the statistical properties of a time series (like mean and variance) do not change over time. If a series is non-stationary, forecasting models struggle because the underlying patterns shift.

The p-value of the initial ADF test determines stationarity. If p-value > 0.05, we accept that it is non-stationary and we apply differencing.

In [ ]:
if p_val > 0.05:
    print("\nSeries is non-stationary. Applying first-order differencing...")
    ts_diff = ts_monthly.diff().dropna()
    print("ADF Test on Differenced Series:")
    run_adf_test(ts_diff)
else:
    print("\nSeries is already stationary.")

## TASK 3: FORECASTING

### Model Evaluation Metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
def mean_absolute_percentage_error(y_true, y_pred): 
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

metrics = {}

# Split data into train and test
train = ts_monthly.iloc[:-6]
test = ts_monthly.iloc[-6:]

### MODEL 1: SARIMA

In [ ]:
import statsmodels.api as sm

# Train SARIMA
sarima_model = sm.tsa.statespace.SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12),
                                         enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False)

sarima_pred = sarima_fit.get_forecast(steps=len(test)).predicted_mean

metrics['SARIMA'] = {
    'MAE': mean_absolute_error(test, sarima_pred),
    'RMSE': np.sqrt(mean_squared_error(test, sarima_pred)),
    'MAPE': mean_absolute_percentage_error(test, sarima_pred)
}

# Retrain on full
sarima_full = sm.tsa.statespace.SARIMAX(ts_monthly, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit(disp=False)
sarima_future = sarima_full.get_forecast(steps=3)
sarima_future_mean = sarima_future.predicted_mean
sarima_ci = sarima_future.conf_int()

plt.figure(figsize=(12, 6))
plt.plot(ts_monthly.index, ts_monthly.values, label='Historical Sales')
plt.plot(sarima_future_mean.index, sarima_future_mean.values, label='SARIMA Forecast', color='red')
plt.fill_between(sarima_ci.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1], color='pink', alpha=0.3, label='Confidence Interval')
plt.title('SARIMA 3-Month Forecast')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
save_chart(plt.gcf(), 'sarima_forecast')
plt.show()

joblib.dump(sarima_full, '../models/sarima_model.pkl')

### MODEL 2: Facebook Prophet

In [ ]:
from prophet import Prophet

prophet_df = pd.DataFrame({'ds': ts_monthly.index, 'y': ts_monthly.values})
prophet_train = prophet_df.iloc[:-6]
prophet_test = prophet_df.iloc[-6:]

# Train
m = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
m.fit(prophet_train)

# Test Eval
future_test = m.make_future_dataframe(periods=len(test), freq='MS')
forecast_test = m.predict(future_test)
prophet_pred = forecast_test['yhat'].iloc[-len(test):].values

metrics['Prophet'] = {
    'MAE': mean_absolute_error(test, prophet_pred),
    'RMSE': np.sqrt(mean_squared_error(test, prophet_pred)),
    'MAPE': mean_absolute_percentage_error(test, prophet_pred)
}

# Retrain on Full
m_full = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
m_full.fit(prophet_df)
future_3m = m_full.make_future_dataframe(periods=3, freq='MS')
forecast_3m = m_full.predict(future_3m)

fig = m_full.plot(forecast_3m)
plt.title('Prophet 3-Month Forecast')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
save_chart(fig, 'prophet_forecast')
plt.show()

fig_comp = m_full.plot_components(forecast_3m)
save_chart(fig_comp, 'prophet_components')
plt.show()

joblib.dump(m_full, '../models/prophet_model.pkl')

### MODEL 3: XGBoost

In [ ]:
import xgboost as xgb

xgb_df = pd.DataFrame({'Sales': ts_monthly})
xgb_df['Lag1'] = xgb_df['Sales'].shift(1)
xgb_df['Lag2'] = xgb_df['Sales'].shift(2)
xgb_df['Lag3'] = xgb_df['Sales'].shift(3)
xgb_df['Rolling_Mean'] = xgb_df['Sales'].shift(1).rolling(window=3).mean()
xgb_df['Rolling_Std'] = xgb_df['Sales'].shift(1).rolling(window=3).std()
xgb_df['Month'] = xgb_df.index.month
xgb_df['Quarter'] = xgb_df.index.quarter
xgb_df['Year'] = xgb_df.index.year

xgb_df.dropna(inplace=True)

X = xgb_df.drop('Sales', axis=1)
y = xgb_df['Sales']

X_train, y_train = X.iloc[:-6], y.iloc[:-6]
X_test, y_test = X.iloc[-6:], y.iloc[-6:]

xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

metrics['XGBoost'] = {
    'MAE': mean_absolute_error(y_test, xgb_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, xgb_pred)),
    'MAPE': mean_absolute_percentage_error(y_test, xgb_pred)
}

# Retrain Full Model
xgb_full = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_full.fit(X, y)

# Generate next 3 months features sequentially
last_data = xgb_df.iloc[-1].copy()
last_sales = last_data['Sales']
future_dates = pd.date_range(start=ts_monthly.index[-1] + pd.DateOffset(months=1), periods=3, freq='MS')

xgb_forecast = []
current_lag1 = last_sales
current_lag2 = last_data['Lag1']
current_lag3 = last_data['Lag2']
hist_sales = list(xgb_df['Sales'].values)

for d in future_dates:
    rmean = np.mean(hist_sales[-3:])
    rstd = np.std(hist_sales[-3:])
    
    x_new = pd.DataFrame([{
        'Lag1': current_lag1,
        'Lag2': current_lag2,
        'Lag3': current_lag3,
        'Rolling_Mean': rmean,
        'Rolling_Std': rstd,
        'Month': d.month,
        'Quarter': d.quarter,
        'Year': d.year
    }])
    
    pred = xgb_full.predict(x_new)[0]
    xgb_forecast.append(pred)
    hist_sales.append(pred)
    current_lag3 = current_lag2
    current_lag2 = current_lag1
    current_lag1 = pred

plt.figure(figsize=(12, 6))
plt.plot(ts_monthly.index, ts_monthly.values, label='Historical Sales')
plt.plot(future_dates, xgb_forecast, label='XGBoost Forecast', color='green', marker='o')
plt.title('XGBoost 3-Month Forecast')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
save_chart(plt.gcf(), 'xgboost_forecast')
plt.show()

joblib.dump(xgb_full, '../models/xgboost_model.pkl')

### Comparison and Recommendation

In [ ]:
metrics_df = pd.DataFrame(metrics).T
print(metrics_df)
metrics_df.to_csv('../outputs/model_metrics.csv')

best_model_name = metrics_df['MAPE'].idxmin()
print(f"\nRecommended Model: {best_model_name} (Lowest MAPE)")

final_forecast_df = forecast_3m[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(3)
final_forecast_df.to_csv('../outputs/final_3m_forecast.csv', index=False)

## TASK 4: CATEGORY & REGION FORECASTING

In [ ]:
segments = [
    ('Category', 'Furniture'),
    ('Category', 'Technology'),
    ('Category', 'Office Supplies'),
    ('Region', 'West'),
    ('Region', 'East')
]

plt.figure(figsize=(15, 8))
segment_forecasts = []

for col, val in segments:
    sub_df = df[df[col] == val].set_index('Order Date').resample('MS')['Sales'].sum().reset_index()
    sub_df.columns = ['ds', 'y']
    m_seg = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    m_seg.fit(sub_df)
    fut = m_seg.make_future_dataframe(periods=3, freq='MS')
    fcst = m_seg.predict(fut)
    
    plt.plot(fcst['ds'], fcst['yhat'], label=f'{val} Forecast')
    
    fcst['Segment'] = val
    segment_forecasts.append(fcst[['ds', 'yhat', 'Segment']].tail(3))

plt.title('3-Month Forecasts by Category and Region')
plt.xlabel('Date')
plt.ylabel('Forecasted Sales ($)')
plt.legend()
save_chart(plt.gcf(), 'segment_forecasts')
plt.show()

pd.concat(segment_forecasts).to_csv('../outputs/segment_forecasts.csv', index=False)

## TASK 5: ANOMALY DETECTION

In [ ]:
from sklearn.ensemble import IsolationForest

weekly_df = df.set_index('Order Date').resample('W')['Sales'].sum().reset_index()

# 1. Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
weekly_df['Iso_Anomaly'] = iso.fit_predict(weekly_df[['Sales']])

plt.figure(figsize=(12, 6))
plt.plot(weekly_df['Order Date'], weekly_df['Sales'], label='Weekly Sales', color='gray')
anomalies_iso = weekly_df[weekly_df['Iso_Anomaly'] == -1]
plt.scatter(anomalies_iso['Order Date'], anomalies_iso['Sales'], color='red', label='Isolation Forest Anomalies')
plt.title('Anomaly Detection: Isolation Forest')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
save_chart(plt.gcf(), 'anomaly_isolation_forest')
plt.show()

# 2. Rolling Z-Score
weekly_df['Rolling_Mean'] = weekly_df['Sales'].rolling(window=4).mean()
weekly_df['Rolling_Std'] = weekly_df['Sales'].rolling(window=4).std()
weekly_df['Z_Score'] = (weekly_df['Sales'] - weekly_df['Rolling_Mean']) / weekly_df['Rolling_Std']

weekly_df['Z_Anomaly'] = np.where(np.abs(weekly_df['Z_Score']) > 2, -1, 1)

plt.figure(figsize=(12, 6))
plt.plot(weekly_df['Order Date'], weekly_df['Sales'], label='Weekly Sales', color='gray')
anomalies_z = weekly_df[weekly_df['Z_Anomaly'] == -1]
plt.scatter(anomalies_z['Order Date'], anomalies_z['Sales'], color='orange', label='Z-Score Anomalies')
plt.title('Anomaly Detection: Rolling Z-Score')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
save_chart(plt.gcf(), 'anomaly_z_score')
plt.show()

weekly_df.to_csv('../outputs/anomaly_report.csv', index=False)

## TASK 6: PRODUCT DEMAND SEGMENTATION

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Aggregate by Sub-category
subcat_df = df.groupby('Sub-Category').agg(
    Sales_Volume=('Sales', 'sum'),
    Order_Count=('Order ID', 'nunique'),
    Sales_Std=('Sales', 'std')
).reset_index()

subcat_df.fillna(0, inplace=True)

# Calculate Growth Rate (Current Year vs Previous Year)
current_year = df['Year'].max()
cy_sales = df[df['Year'] == current_year].groupby('Sub-Category')['Sales'].sum()
py_sales = df[df['Year'] == current_year - 1].groupby('Sub-Category')['Sales'].sum()

growth = (cy_sales - py_sales) / py_sales
subcat_df['Growth_Rate'] = subcat_df['Sub-Category'].map(growth).fillna(0)

subcat_df['Average_Order_Value'] = subcat_df['Sales_Volume'] / subcat_df['Order_Count']
subcat_df['Sales_Volatility'] = subcat_df['Sales_Std'] / subcat_df['Average_Order_Value']
subcat_df.fillna(0, inplace=True)

# Standardize
features = ['Sales_Volume', 'Growth_Rate', 'Sales_Volatility', 'Average_Order_Value']
scaler = StandardScaler()
scaled_features = scaler.fit_transform(subcat_df[features])

kmeans = KMeans(n_clusters=4, random_state=42)
subcat_df['Cluster'] = kmeans.fit_predict(scaled_features)

pca = PCA(n_components=2)
pca_res = pca.fit_transform(scaled_features)
subcat_df['PCA1'] = pca_res[:, 0]
subcat_df['PCA2'] = pca_res[:, 1]

plt.figure(figsize=(10, 8))
sns.scatterplot(data=subcat_df, x='PCA1', y='PCA2', hue='Cluster', palette='Set1', s=100)
for i, txt in enumerate(subcat_df['Sub-Category']):
    plt.annotate(txt, (subcat_df['PCA1'].iloc[i]+0.1, subcat_df['PCA2'].iloc[i]), fontsize=8)
plt.title('Product Demand Segmentation (PCA 2D)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Cluster')
save_chart(plt.gcf(), 'demand_segmentation')
plt.show()

def assign_label(cluster):
    if cluster == 0: return 'Steady & Low Value'
    elif cluster == 1: return 'High Growth, High Value'
    elif cluster == 2: return 'Volatile / Seasonal'
    else: return 'Cash Cows (High Volume)'

subcat_df['Business_Label'] = subcat_df['Cluster'].apply(assign_label)
subcat_df.to_csv('../outputs/demand_segmentation.csv', index=False)
subcat_df[['Sub-Category', 'Business_Label']]